In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Splitting Data into Training and Testing Sets (Time-Based)

In [ ]:
import pandas as pd
import os

# Define the path to the cleaned data file
cleaned_data_path = '/content/drive/MyDrive/citibike_project/Rabin/CleanedData.csv'

# Load the cleaned data
clean_df = pd.read_csv(cleaned_data_path)

# Ensure 'started_at' is a datetime object for time-based splitting
clean_df['started_at'] = pd.to_datetime(clean_df['started_at'])

print(f"Successfully loaded cleaned data from: {cleaned_data_path}")
print(f"Shape of clean_df: {clean_df.shape}")

# Display the first few rows of the loaded DataFrame
print("\nHead of the loaded cleaned data:")
display(clean_df.head())

Successfully loaded cleaned data from: /content/drive/MyDrive/citibike_project/Rabin/CleanedData.csv
Shape of clean_df: (1002333, 14)

Head of the loaded cleaned data:


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_mins
0,FBA120ADE8D22E7D,electric_bike,2025-12-26 10:20:09.188,2025-12-26 10:40:13.508,Lafayette Park,JC078,Vesey St & Greenwich St,5216.07,40.713464,-74.062859,40.712547,-74.011131,casual,20.072000
1,DB8F52B8AE51DD28,electric_bike,2025-12-16 06:48:28.111,2025-12-16 06:56:10.969,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.714300
2,C16D1CA1746E065F,electric_bike,2025-12-20 12:50:55.361,2025-12-20 12:58:32.234,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.614550
3,7A613AF4BD6D75B9,electric_bike,2025-12-22 16:59:42.922,2025-12-22 17:04:29.327,JC Medical Center,JC110,Exchange Pl,JC116,40.715391,-74.049692,40.716366,-74.034344,casual,4.773417
4,2448F2B5B85E52B1,electric_bike,2025-12-17 18:06:09.352,2025-12-17 18:14:13.548,Baldwin at Montgomery,JC020,Exchange Pl,JC116,40.723659,-74.064194,40.716366,-74.034344,member,8.069933


### Splitting Cleaned Trip Data into Training, Validation, and Testing Sets (Time-Based)

In [ ]:
import pandas as pd
import numpy as np

# Get all unique dates from the 'started_at' column and sort them
all_dates = clean_df['started_at'].dt.date.sort_values().unique()
N_total_unique_dates = len(all_dates)

# Define the test set to be exactly the last 7 unique days
test_days_count = 7
if N_total_unique_dates < test_days_count:
    raise ValueError(f"Not enough unique dates ({N_total_unique_dates}) for a {test_days_count}-day test set.")

# The boundary for the start of the test set
test_start_date_boundary = all_dates[N_total_unique_dates - test_days_count]

# Dates available for training and validation are those *before* the test_start_date_boundary
dates_for_train_val = all_dates[all_dates < test_start_date_boundary]
N_train_val_unique_dates = len(dates_for_train_val)

# Now, split the remaining dates (dates_for_train_val) into training and validation.
# Using a 70/30 split for train/val from these remaining dates.
train_val_split_ratio = 0.70 # 70% for train, 30% for validation within N_train_val_unique_dates

train_split_idx = int(N_train_val_unique_dates * train_val_split_ratio)

# The boundary for the start of the validation set (and end of the training set)
train_date_boundary = dates_for_train_val[train_split_idx]


print(f"Train data ends before: {train_date_boundary}")
print(f"Validation data starts from: {train_date_boundary} and ends before: {test_start_date_boundary}")
print(f"Testing data starts from: {test_start_date_boundary} and includes the last {test_days_count} unique dates.")


# Perform the time-based split on the original clean_df
train_df_raw = clean_df[clean_df['started_at'].dt.date < train_date_boundary]
val_df_raw = clean_df[(clean_df['started_at'].dt.date >= train_date_boundary) & (clean_df['started_at'].dt.date < test_start_date_boundary)]
test_df_raw = clean_df[clean_df['started_at'].dt.date >= test_start_date_boundary]

print(f"\nRaw Training data shape: {train_df_raw.shape}")
print(f"Raw Validation data shape: {val_df_raw.shape}")
print(f"Raw Testing data shape: {test_df_raw.shape}")

print("\nHead of Raw Training Data (first 5 rows):")
display(train_df_raw.head())

print("\nHead of Raw Validation Data (first 5 rows):")
display(val_df_raw.head())

print("\nHead of Raw Testing Data (first 5 rows):")
display(test_df_raw.head())

Train data ends before: 2025-09-08
Validation data starts from: 2025-09-08 and ends before: 2025-12-25
Testing data starts from: 2025-12-25 and includes the last 7 unique dates.

Raw Training data shape: (681166, 14)
Raw Validation data shape: (315860, 14)
Raw Testing data shape: (5307, 14)

Head of Raw Training Data (first 5 rows):


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_mins
48417,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,6.192900
48418,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,11.461350
48419,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,21.377617
48420,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,22.934333
48421,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,25.822100



Head of Raw Validation Data (first 5 rows):


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_mins
1,DB8F52B8AE51DD28,electric_bike,2025-12-16 06:48:28.111,2025-12-16 06:56:10.969,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.714300
2,C16D1CA1746E065F,electric_bike,2025-12-20 12:50:55.361,2025-12-20 12:58:32.234,Dr. Lena Edwards Park,JC117,Exchange Pl,JC116,40.713203,-74.058261,40.716366,-74.034344,member,7.614550
3,7A613AF4BD6D75B9,electric_bike,2025-12-22 16:59:42.922,2025-12-22 17:04:29.327,JC Medical Center,JC110,Exchange Pl,JC116,40.715391,-74.049692,40.716366,-74.034344,casual,4.773417
4,2448F2B5B85E52B1,electric_bike,2025-12-17 18:06:09.352,2025-12-17 18:14:13.548,Baldwin at Montgomery,JC020,Exchange Pl,JC116,40.723659,-74.064194,40.716366,-74.034344,member,8.069933
5,66BCA70446052D6E,classic_bike,2025-12-05 19:16:10.581,2025-12-05 19:25:15.326,Montgomery St,JC099,Exchange Pl,JC116,40.719420,-74.050990,40.716366,-74.034344,member,9.079083



Head of Raw Testing Data (first 5 rows):


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_mins
0,FBA120ADE8D22E7D,electric_bike,2025-12-26 10:20:09.188,2025-12-26 10:40:13.508,Lafayette Park,JC078,Vesey St & Greenwich St,5216.07,40.713464,-74.062859,40.712547,-74.011131,casual,20.072000
8,111402FD8743ACF8,electric_bike,2025-12-27 13:08:05.701,2025-12-27 13:11:31.374,Columbus Drive,JC014,Exchange Pl,JC116,40.718355,-74.038914,40.716366,-74.034344,member,3.427883
10,22ABFD90D333881C,classic_bike,2025-12-29 18:43:21.803,2025-12-29 18:53:16.691,Brunswick & 6th,JC081,Exchange Pl,JC116,40.726012,-74.050389,40.716366,-74.034344,member,9.914800
39,1DE13EEB413820FE,electric_bike,2025-12-31 17:10:30.242,2025-12-31 18:09:10.672,Exchange Pl,JC116,10 Ave & W 14 St,6157.04,40.716366,-74.034344,40.741982,-74.008316,casual,58.673833
75,54D78E16B6E0223B,electric_bike,2025-12-25 08:56:39.927,2025-12-25 08:59:59.349,Brunswick & 6th,JC081,Newark Ave,JC032,40.726012,-74.050389,40.721525,-74.046305,casual,3.323700


In [ ]:
# --- STEP 4: MASTER NET FLOW AGGREGATION & COORDINATE MAPPING ---

# Ensure 'hour' column is available for aggregation
clean_df['hour'] = clean_df['started_at'].dt.hour

# 1. Aggregate Outflows (Departures)
outflows = clean_df.groupby(['start_station_name', 'hour']).size().reset_index(name='outflow')
outflows.columns = ['station', 'hour', 'outflow']

# 2. Aggregate Inflows (Arrivals)
inflows = clean_df.groupby(['end_station_name', 'hour']).size().reset_index(name='inflow')
inflows.columns = ['station', 'hour', 'inflow']

# 3. Create the net_flow_df by merging Inflows and Outflows
net_flow_df = pd.merge(outflows, inflows, on=['station', 'hour'], how='outer').fillna(0)
net_flow_df['net_demand'] = net_flow_df['inflow'] - net_flow_df['outflow']

# 4. Create a ROBUST Coordinate Lookup (from both start and end stations)
start_coords = clean_df[['start_station_name', 'start_lat', 'start_lng']].rename(
    columns={'start_station_name': 'station', 'start_lat': 'lat', 'start_lng': 'lng'}
)
end_coords = clean_df[['end_station_name', 'end_lat', 'end_lng']].rename(
    columns={'end_station_name': 'station', 'end_lat': 'lat', 'end_lng': 'lng'}
)

# Combine and average to get one lat/lng per station name
coords_lookup = pd.concat([start_coords, end_coords]).dropna(subset=['station']).groupby('station')[['lat', 'lng']].mean().reset_index()

# 5. Merge the coordinates into our final dataframe
net_flow_df = pd.merge(net_flow_df, coords_lookup, on='station', how='left')

# Check the result
print("Net Flow Table successfully created!")
print(f"Total stations processed: {net_flow_df['station'].nunique()}")
display(net_flow_df.head())

Net Flow Table successfully created!
Total stations processed: 487


,station,hour,outflow,inflow,net_demand,lat,lng
0,1 Ave & E 38 St,8,0.0,1.0,1.0,40.746202,-73.971822
1,1 Ave & E 38 St,11,0.0,1.0,1.0,40.746202,-73.971822
2,1 Ave & E 6 St,10,0.0,1.0,1.0,40.726331,-73.986169
3,1 Ave & E 6 St,15,0.0,2.0,2.0,40.726331,-73.986169
4,1 Ave & E 6 St,16,0.0,1.0,1.0,40.726331,-73.986169


In [ ]:
import pandas as pd
import numpy as np

def feature_engineer_for_split(df_raw_split, global_coords_lookup):
    # Make a copy to avoid SettingWithCopyWarning
    df_processed = df_raw_split.copy()

    # Derive 'date' and 'hour' from 'started_at' for aggregation
    df_processed['date'] = df_processed['started_at'].dt.date
    df_processed['hour'] = df_processed['started_at'].dt.hour

    # --- 1. Calculate Net Demand for this split (per station, per date, per hour) ---
    # Aggregate Outflows (Departures)
    outflows = df_processed.groupby(['start_station_name', 'date', 'hour']).size().reset_index(name='outflow')
    outflows.columns = ['station', 'date', 'hour', 'outflow']

    # Aggregate Inflows (Arrivals)
    inflows = df_processed.groupby(['end_station_name', 'date', 'hour']).size().reset_index(name='inflow')
    inflows.columns = ['station', 'date', 'hour', 'inflow']

    # Merge Inflows and Outflows for this split
    current_net_flow_df = pd.merge(outflows, inflows, on=['station', 'date', 'hour'], how='outer').fillna(0)
    current_net_flow_df['net_demand'] = current_net_flow_df['inflow'] - current_net_flow_df['outflow']

    # --- 2. PREPARING THE COMPLETE TIME-SERIES GRID (RE-INDEXING) ---
    # Get unique stations, dates, and hours from the current net flow
    stations_in_split = current_net_flow_df['station'].unique()
    dates_in_split = current_net_flow_df['date'].unique()
    hours_in_split = range(24) # All 24 hours to ensure continuous series

    # Create a MultiIndex of all combinations for this split
    multi_index = pd.MultiIndex.from_product(
        [stations_in_split, dates_in_split, hours_in_split],
        names=['station', 'date', 'hour']
    )

    # Create the master grid for this split
    feature_grid_split = pd.DataFrame(index=multi_index).reset_index()
    feature_grid_split['date'] = pd.to_datetime(feature_grid_split['date']) # Ensure date is datetime64[ns]

    # FIX: Ensure 'date' column in current_net_flow_df is also datetime64[ns] for merging
    current_net_flow_df['date'] = pd.to_datetime(current_net_flow_df['date'])

    # Merge with the current split's net_demand
    # Fill NaNs with 0 where there was no activity at a particular station/date/hour
    final_df_split = pd.merge(
        feature_grid_split,
        current_net_flow_df[['station', 'date', 'hour', 'net_demand']],
        on=['station', 'date', 'hour'],
        how='left'
    ).fillna({'net_demand': 0})

    # Merge coordinates using the global_coords_lookup
    # Drop existing lat/lng if they somehow ended up here before merging to avoid conflicts
    final_df_split = final_df_split.drop(columns=['lat', 'lng'], errors='ignore')
    final_df_split = pd.merge(final_df_split, global_coords_lookup, on='station', how='left')

    # 3. CYCLICAL TIME ENCODING
    final_df_split['hour_sin'] = np.sin(2 * np.pi * final_df_split['hour'] / 24)
    final_df_split['hour_cos'] = np.cos(2 * np.pi * final_df_split['hour'] / 24)

    final_df_split['day_of_week'] = final_df_split['date'].dt.dayofweek
    final_df_split['day_sin'] = np.sin(2 * np.pi * final_df_split['day_of_week'] / 7)
    final_df_split['day_cos'] = np.cos(2 * np.pi * final_df_split['day_of_week'] / 7)

    # --- 4. MULTI-SCALE LAG FEATURES ---
    # Sort by station and time to ensure shift() works correctly
    final_df_split = final_df_split.sort_values(['station', 'date', 'hour'])

    lags = [1, 2, 3, 24]
    for lag in lags:
        # Ensure lags are calculated within each station group
        final_df_split[f'lag_{lag}h'] = final_df_split.groupby('station')['net_demand'].shift(lag)

    # 5. ROLLING WINDOW STATISTICS
    # 3-hour moving average of demand, shifted by 1 to prevent data leakage
    final_df_split['rolling_mean_3h'] = final_df_split.groupby('station')['net_demand'].transform(
        lambda x: x.shift(1).rolling(window=3, min_periods=1).mean() # min_periods=1 to allow calculation with fewer points at start
    )

    # --- 6. CLEANUP ---
    # Drop rows that have NaN values due to lag features at the beginning of each time series for each station
    final_df_split = final_df_split.dropna()

    return final_df_split

In [ ]:
print("Applying feature engineering to training, validation, and testing sets...")

train_df = feature_engineer_for_split(train_df_raw, coords_lookup)
val_df = feature_engineer_for_split(val_df_raw, coords_lookup)
test_df = feature_engineer_for_split(test_df_raw, coords_lookup)

print("Feature Engineering Complete for all splits!")
print(f"\nTraining data shape (feature-engineered): {train_df.shape}")
print(f"Validation data shape (feature-engineered): {val_df.shape}")
print(f"Testing data shape (feature-engineered): {test_df.shape}")

print("\nHead of Feature-Engineered Training Data:")
display(train_df.head())

print("\nHead of Feature-Engineered Validation Data:")
display(val_df.head())

print("\nHead of Feature-Engineered Testing Data:")
display(test_df.head())

Applying feature engineering to training, validation, and testing sets...
Feature Engineering Complete for all splits!

Training data shape (feature-engineered): (2466000, 16)
Validation data shape (feature-engineered): (857712, 16)
Testing data shape (feature-engineered): (16416, 16)

Head of Feature-Engineered Training Data:


,station,date,hour,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
1632,1 Ave & E 38 St,2025-01-01,0,0.0,40.746202,-73.971822,0.000000,1.000000,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1633,1 Ave & E 38 St,2025-01-01,1,0.0,40.746202,-73.971822,0.258819,0.965926,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1634,1 Ave & E 38 St,2025-01-01,2,0.0,40.746202,-73.971822,0.500000,0.866025,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1635,1 Ave & E 38 St,2025-01-01,3,0.0,40.746202,-73.971822,0.707107,0.707107,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1636,1 Ave & E 38 St,2025-01-01,4,0.0,40.746202,-73.971822,0.866025,0.500000,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0



Head of Feature-Engineered Validation Data:


,station,date,hour,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
192,1 Ave & E 6 St,2025-09-09,0,0.0,40.726331,-73.986169,0.000000,1.000000,1,0.781831,0.62349,0.0,0.0,0.0,0.0,0.0
193,1 Ave & E 6 St,2025-09-09,1,0.0,40.726331,-73.986169,0.258819,0.965926,1,0.781831,0.62349,0.0,0.0,0.0,0.0,0.0
194,1 Ave & E 6 St,2025-09-09,2,0.0,40.726331,-73.986169,0.500000,0.866025,1,0.781831,0.62349,0.0,0.0,0.0,0.0,0.0
195,1 Ave & E 6 St,2025-09-09,3,0.0,40.726331,-73.986169,0.707107,0.707107,1,0.781831,0.62349,0.0,0.0,0.0,0.0,0.0
196,1 Ave & E 6 St,2025-09-09,4,0.0,40.726331,-73.986169,0.866025,0.500000,1,0.781831,0.62349,0.0,0.0,0.0,0.0,0.0



Head of Feature-Engineered Testing Data:


,station,date,hour,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
48,10 Ave & W 14 St,2025-12-26,0,0.0,40.741982,-74.008316,0.000000,1.000000,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
49,10 Ave & W 14 St,2025-12-26,1,0.0,40.741982,-74.008316,0.258819,0.965926,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
50,10 Ave & W 14 St,2025-12-26,2,0.0,40.741982,-74.008316,0.500000,0.866025,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
51,10 Ave & W 14 St,2025-12-26,3,0.0,40.741982,-74.008316,0.707107,0.707107,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
52,10 Ave & W 14 St,2025-12-26,4,0.0,40.741982,-74.008316,0.866025,0.500000,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0


In [ ]:
import os

# Define the directory to save the split datasets
output_dir = os.path.join(extraction_path, 'ProcessedData')
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory for processed data: {output_dir}")

# Save the training set
train_output_path = os.path.join(output_dir, 'train_set_processed.csv')
train_df.to_csv(train_output_path, index=False)
print(f"Training set saved to: {train_output_path}")

# Save the validation set
val_output_path = os.path.join(output_dir, 'val_set_processed.csv')
val_df.to_csv(val_output_path, index=False)
print(f"Validation set saved to: {val_output_path}")

# Save the testing set
test_output_path = os.path.join(output_dir, 'test_set_processed.csv')
test_df.to_csv(test_output_path, index=False)
print(f"Testing set saved to: {test_output_path}")

print("All feature-engineered datasets saved successfully!")

NameError: name 'extraction_path' is not defined

You can now load these datasets directly for model training as needed:

```python
import pandas as pd
import os

output_dir = '/content/drive/MyDrive/citibike_project/Rabin/ProcessedData' # Adjust path if needed

loaded_train_df = pd.read_csv(os.path.join(output_dir, 'train_set_processed.csv'))
loaded_val_df = pd.read_csv(os.path.join(output_dir, 'val_set_processed.csv'))
loaded_test_df = pd.read_csv(os.path.join(output_dir, 'test_set_processed.csv'))

print("Loaded DataFrames:")
print(f"Loaded train_df shape: {loaded_train_df.shape}")
print(f"Loaded val_df shape: {loaded_val_df.shape}")
print(f"Loaded test_df shape: {loaded_test_df.shape}")
```

In [ ]:
import pandas as pd
import os

# Define the base extraction path (from previous cells)
extraction_path = '/content/drive/MyDrive/citibike_project/Rabin'

# Construct the full path to the specified training data file
training_data_path = os.path.join(extraction_path, 'CleanedData/micro_mobility_training_data_2025.csv')

# Load the training data
mobility_df = pd.read_csv(training_data_path)

print(f"Successfully loaded data from: {training_data_path}")
print(f"Shape of mobility_df: {mobility_df.shape}")

# Display the first few rows of the loaded DataFrame
print("\nHead of the loaded training data:")
display(mobility_df.head())

Successfully loaded data from: /content/drive/MyDrive/citibike_project/Rabin/CleanedData/micro_mobility_training_data_2025.csv
Shape of mobility_df: (4254919, 18)

Head of the loaded training data:


,station,date,hour,datetime_hour,is_weekend,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
0,1 Ave & E 38 St,2025-01-01,23,2025-01-01 23:00:00,0,0.0,40.746202,-73.971822,-0.258819,0.965926,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1,1 Ave & E 38 St,2025-01-02,0,2025-01-02 00:00:00,0,0.0,40.746202,-73.971822,0.000000,1.000000,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
2,1 Ave & E 38 St,2025-01-02,1,2025-01-02 01:00:00,0,0.0,40.746202,-73.971822,0.258819,0.965926,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
3,1 Ave & E 38 St,2025-01-02,2,2025-01-02 02:00:00,0,0.0,40.746202,-73.971822,0.500000,0.866025,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
4,1 Ave & E 38 St,2025-01-02,3,2025-01-02 03:00:00,0,0.0,40.746202,-73.971822,0.707107,0.707107,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0


### Splitting Data into Training, Validation, and Testing Sets (Time-Based)

In [ ]:
import pandas as pd

# Ensure the 'date' column is in datetime format for proper sorting
mobility_df['date'] = pd.to_datetime(mobility_df['date'])

# Get all unique dates and sort them
all_dates = mobility_df['date'].sort_values().unique()

# Define split percentages (e.g., 70% train, 15% validation, 15% test)
train_split_idx = int(len(all_dates) * 0.70)
val_split_idx = int(len(all_dates) * 0.85) # 70% + 15%

# Get the split dates
train_date = all_dates[train_split_idx]
val_date = all_dates[val_split_idx]

print(f"Splitting data for training before: {train_date}")
print(f"Splitting data for validation before: {val_date}")

# Perform the time-based split
train_df = mobility_df[mobility_df['date'] < train_date]
val_df = mobility_df[(mobility_df['date'] >= train_date) & (mobility_df['date'] < val_date)]
test_df = mobility_df[mobility_df['date'] >= val_date]

print(f"\nTraining data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")
print(f"Testing data shape: {test_df.shape}")

print("\nHead of Training Data:")
display(train_df.head())

print("\nHead of Validation Data:")
display(val_df.head())

print("\nHead of Testing Data:")
display(test_df.head())

Splitting data for training before: 2025-09-13 00:00:00
Splitting data for validation before: 2025-11-07 00:00:00

Training data shape: (2969239, 18)
Validation data shape: (642840, 18)
Testing data shape: (642840, 18)

Head of Training Data:


,station,date,hour,datetime_hour,is_weekend,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
0,1 Ave & E 38 St,2025-01-01,23,2025-01-01 23:00:00,0,0.0,40.746202,-73.971822,-0.258819,0.965926,2,0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
1,1 Ave & E 38 St,2025-01-02,0,2025-01-02 00:00:00,0,0.0,40.746202,-73.971822,0.000000,1.000000,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
2,1 Ave & E 38 St,2025-01-02,1,2025-01-02 01:00:00,0,0.0,40.746202,-73.971822,0.258819,0.965926,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
3,1 Ave & E 38 St,2025-01-02,2,2025-01-02 02:00:00,0,0.0,40.746202,-73.971822,0.500000,0.866025,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
4,1 Ave & E 38 St,2025-01-02,3,2025-01-02 03:00:00,0,0.0,40.746202,-73.971822,0.707107,0.707107,3,0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0



Head of Validation Data:


,station,date,hour,datetime_hour,is_weekend,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
6097,1 Ave & E 38 St,2025-09-13,0,2025-09-13 00:00:00,1,0.0,40.746202,-73.971822,0.000000,1.000000,5,-0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
6098,1 Ave & E 38 St,2025-09-13,1,2025-09-13 01:00:00,1,0.0,40.746202,-73.971822,0.258819,0.965926,5,-0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
6099,1 Ave & E 38 St,2025-09-13,2,2025-09-13 02:00:00,1,0.0,40.746202,-73.971822,0.500000,0.866025,5,-0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
6100,1 Ave & E 38 St,2025-09-13,3,2025-09-13 03:00:00,1,0.0,40.746202,-73.971822,0.707107,0.707107,5,-0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0
6101,1 Ave & E 38 St,2025-09-13,4,2025-09-13 04:00:00,1,0.0,40.746202,-73.971822,0.866025,0.500000,5,-0.974928,-0.222521,0.0,0.0,0.0,0.0,0.0



Head of Testing Data:


,station,date,hour,datetime_hour,is_weekend,net_demand,lat,lng,hour_sin,hour_cos,day_of_week,day_sin,day_cos,lag_1h,lag_2h,lag_3h,lag_24h,rolling_mean_3h
7417,1 Ave & E 38 St,2025-11-07,0,2025-11-07 00:00:00,0,0.0,40.746202,-73.971822,0.000000,1.000000,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
7418,1 Ave & E 38 St,2025-11-07,1,2025-11-07 01:00:00,0,0.0,40.746202,-73.971822,0.258819,0.965926,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
7419,1 Ave & E 38 St,2025-11-07,2,2025-11-07 02:00:00,0,0.0,40.746202,-73.971822,0.500000,0.866025,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
7420,1 Ave & E 38 St,2025-11-07,3,2025-11-07 03:00:00,0,0.0,40.746202,-73.971822,0.707107,0.707107,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0
7421,1 Ave & E 38 St,2025-11-07,4,2025-11-07 04:00:00,0,0.0,40.746202,-73.971822,0.866025,0.500000,4,-0.433884,-0.900969,0.0,0.0,0.0,0.0,0.0


In [ ]:
import os

# Define the path to the directory containing the compressed files
dataset_path = '/content/drive/MyDrive/citibike_project/dataset'

# Define the target directory for extraction
extraction_path = '/content/drive/MyDrive/citibike_project/Rabin'

# Create the target directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)
print(f"Target extraction directory '{extraction_path}' ensured.")

Target extraction directory '/content/drive/MyDrive/citibike_project/Rabin' ensured.


In [ ]:
import zipfile

In [ ]:
print(f"Listing contents of '{dataset_path}':")
for item in os.listdir(dataset_path):
    if item.endswith('.zip'):
        zip_file_path = os.path.join(dataset_path, item)
        print(f"Extracting '{item}' to '{extraction_path}'...")
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_path)
        print(f"Successfully extracted '{item}'.")

print(f"All compressed files extracted to '{extraction_path}'.")
print(f"Contents of extraction directory '{extraction_path}': {os.listdir(extraction_path)}")

Listing contents of '/content/drive/MyDrive/citibike_project/dataset':
Extracting 'JC-202512-citibike-tripdata.csv.zip' to '/content/drive/MyDrive/citibike_project/Rabin'...


NameError: name 'zipfile' is not defined

In [ ]:
import pandas as pd

# List all files within the extraction_path directory
all_files = os.listdir(extraction_path)

# Filter to include only .csv files
csv_files = [f for f in all_files if f.endswith('.csv')]

# Create an empty list to store the loaded DataFrames
dataframes = []

print(f"Found CSV files: {csv_files}")

# Iterate through the filtered list of CSV files and load them into DataFrames
for csv_file in csv_files:
    file_path = os.path.join(extraction_path, csv_file)
    print(f"Loading {csv_file}...")
    df = pd.read_csv(file_path)
    dataframes.append(df)
    print(f"Successfully loaded {csv_file} with {len(df)} rows and {len(df.columns)} columns.")

# Print the first 5 rows of each DataFrame
for i, df in enumerate(dataframes):
    print(f"\n--- Head of DataFrame {i+1} ({csv_files[i]}) ---")
    print(df.head())

Found CSV files: ['JC-202512-citibike-tripdata.csv', 'JC-202501-citibike-tripdata.csv', 'JC-202503-citibike-tripdata.csv', 'JC-202504-citibike-tripdata.csv', 'JC-202505-citibike-tripdata.csv', 'JC-202506-citibike-tripdata.csv', 'JC-202507-citibike-tripdata.csv', 'JC-202508-citibike-tripdata.csv', 'JC-202509-citibike-tripdata.csv', 'JC-202510-citibike-tripdata.csv', 'JC-202511-citibike-tripdata.csv', 'JC-202502-citibike-tripdata.csv', 'CleanedData.csv']
Loading JC-202512-citibike-tripdata.csv...
Successfully loaded JC-202512-citibike-tripdata.csv with 48474 rows and 13 columns.
Loading JC-202501-citibike-tripdata.csv...
Successfully loaded JC-202501-citibike-tripdata.csv with 50611 rows and 13 columns.
Loading JC-202503-citibike-tripdata.csv...
Successfully loaded JC-202503-citibike-tripdata.csv with 73293 rows and 13 columns.
Loading JC-202504-citibike-tripdata.csv...
Successfully loaded JC-202504-citibike-tripdata.csv with 81553 rows and 13 columns.
Loading JC-202505-citibike-tripdata

In [ ]:
combined_df = pd.concat(dataframes, ignore_index=True)

print("--- Head of Combined DataFrame ---")
print(combined_df.head())

print(f"\nShape of combined_df: {combined_df.shape}")

In [ ]:
print("--- Data Info (Data Types and Non-Null Counts) ---")
combined_df.info()

print("\n--- Missing Values Count ---")
print(combined_df.isnull().sum())

print("\n--- Descriptive Statistics for Numerical Columns ---")
print(combined_df.describe())

In [ ]:
import pandas as pd

# Ensure 'started_at' and 'ended_at' are datetime objects
combined_df['started_at'] = pd.to_datetime(combined_df['started_at'])
combined_df['ended_at'] = pd.to_datetime(combined_df['ended_at'])

# Calculate duration in minutes
combined_df['duration_mins'] = (combined_df['ended_at'] - combined_df['started_at']).dt.total_seconds() / 60.0

# Filter out bad data to create clean_df
clean_df = combined_df[(combined_df['duration_mins'] >= 1) & (combined_df['duration_mins'] <= 1440)]

output_path = '/content/drive/MyDrive/citibike_project/Rabin/CleanedData.csv'
clean_df.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")

In [ ]:
print("--- Head of Cleaned DataFrame (clean_df) ---")
display(clean_df.head())
print(f"\nShape of clean_df: {clean_df.shape}")

**Step 1: Data Cleaning & Feature Extraction (The Basics)**

Before visualizing anything, you must clean the anomalies and create the features your model will need.

Datetime Conversion: Convert started_at and ended_at from strings (objects) to Pandas Datetime objects.

Calculate Trip Duration: Subtract started_at from ended_at to get the duration in minutes.

Filter Anomalies: Drop trips that are under 60 seconds (usually false starts or broken bikes) and trips over 24 hours (stolen/lost bikes).

Handle Missing Values: You have missing end_station_name and end_lat/end_lng. With e-bikes, riders sometimes lock them outside of official stations. Decide whether to drop these or categorize them as "Out of Station".

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Convert to Datetime
combined_df['started_at'] = pd.to_datetime(combined_df['started_at'])
combined_df['ended_at'] = pd.to_datetime(combined_df['ended_at'])

# 2. Calculate Duration in Minutes
combined_df['duration_mins'] = (combined_df['ended_at'] - combined_df['started_at']).dt.total_seconds() / 60.0

# 3. Filter out bad data (negative times, fake starts)
clean_df = combined_df[(combined_df['duration_mins'] >= 1) & (combined_df['duration_mins'] <= 1440)]

# 4. Extract Time Features
clean_df['hour'] = clean_df['started_at'].dt.hour
clean_df['day_of_week'] = clean_df['started_at'].dt.day_name()

# 5. Plot Hourly Demand (Commuter vs Leisure)
plt.figure(figsize=(12, 6))
sns.countplot(data=clean_df, x='hour', hue='member_casual', palette='viridis')
plt.title('Hourly Bike Demand: Members vs Casual Riders')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Convert to Datetime
combined_df['started_at'] = pd.to_datetime(combined_df['started_at'])
combined_df['ended_at'] = pd.to_datetime(combined_df['ended_at'])

# 2. Calculate Duration in Minutes
combined_df['duration_mins'] = (combined_df['ended_at'] - combined_df['started_at']).dt.total_seconds() / 60.0

# 3. Filter out bad data (negative times, fake starts)
clean_df = combined_df[(combined_df['duration_mins'] >= 1) & (combined_df['duration_mins'] <= 1440)]

# 4. Extract Time Features
clean_df['hour'] = clean_df['started_at'].dt.hour
clean_df['day_of_week'] = clean_df['started_at'].dt.day_name()

# 5. Plot Hourly Demand (Commuter vs Leisure)
plt.figure(figsize=(12, 6))
sns.countplot(data=clean_df, x='hour', hue='member_casual', palette='viridis')
plt.title('Hourly Bike Demand: Members vs Casual Riders')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Convert to Datetime
combined_df['started_at'] = pd.to_datetime(combined_df['started_at'])
combined_df['ended_at'] = pd.to_datetime(combined_df['ended_at'])

# 2. Calculate Duration in Minutes
combined_df['duration_mins'] = (combined_df['ended_at'] - combined_df['started_at']).dt.total_seconds() / 60.0

# 3. Filter out bad data (negative times, fake starts)
clean_df = combined_df[(combined_df['duration_mins'] >= 1) & (combined_df['duration_mins'] <= 1440)]

# 4. Extract Time Features
clean_df['hour'] = clean_df['started_at'].dt.hour
clean_df['day_of_week'] = clean_df['started_at'].dt.day_name()

# 5. Plot Hourly Demand (Commuter vs Leisure)
plt.figure(figsize=(12, 6))
sns.countplot(data=clean_df, x='hour', hue='member_casual', palette='viridis')
plt.title('Hourly Bike Demand: Members vs Casual Riders')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.show()

KDE PLOT

In [ ]:
plt.figure(figsize=(12, 7))
sns.kdeplot(combined_df['duration_mins'], label='Raw Data', fill=True, color='blue', alpha=0.5)
sns.kdeplot(clean_df['duration_mins'], label='Cleaned Data', fill=True, color='green', alpha=0.5)
plt.title('Distribution of Trip Duration: Raw vs. Cleaned Data')
plt.xlabel('Trip Duration (minutes)')
plt.ylabel('Density')
plt.xlim(-10, 150) # Limit x-axis for better visualization, as most relevant trips are shorter
plt.legend()
plt.show()

### Insights from Duration Distribution Comparison

The Kernel Density Estimate (KDE) plots effectively highlight the impact of data cleaning on the `duration_mins` distribution:

*   **Raw Data (Blue Curve):** The raw data shows a very long tail extending to high duration values, indicating the presence of many extremely long trips. The peak of the distribution is heavily skewed towards shorter durations, but the long tail suggests outliers that could distort analysis.
*   **Cleaned Data (Green Curve):** After cleaning (filtering out trips under 1 minute and over 24 hours), the distribution becomes much more concentrated and interpretable. The long tail is significantly reduced, and the distribution is much more focused on typical trip durations. This demonstrates that the cleaning process successfully removed anomalous data points, resulting in a more realistic and actionable representation of trip durations.

This comparison confirms the effectiveness of the data cleaning steps in refining the `duration_mins` feature, making it more suitable for further analysis and modeling.

In [ ]:
plt.figure(figsize=(14, 7))
sns.countplot(data=clean_df, x='day_of_week', hue='member_casual', palette='magma', order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
plt.title('Trip Count by Day of Week: Members vs. Casual Riders')
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.legend(title='Rider Type')
plt.show()

### Insights from Daily Demand Patterns (Day of Week)

The count plot showing trips by 'day_of_week' for members and casual riders provides valuable temporal insights:

*   **Member Riders:** Consistent demand throughout weekdays (Monday-Friday), with a slight dip on weekends. This suggests that members predominantly use bikes for commuting or regular weekday activities.
*   **Casual Riders:** Significantly higher usage during weekends (Saturday and Sunday) compared to weekdays. This indicates that casual riders are more likely to use the service for leisure activities.
*   **Overall Trend:** Weekends generally see higher total ridership, driven primarily by casual users, while weekdays maintain a strong base of member riders.

These temporal patterns are crucial for rebalancing efforts. For instance, stations near business districts might experience high demand during weekday mornings and evenings (driven by members), while recreational areas might see peak demand during weekend afternoons (driven by casual riders). Understanding these differences allows for more targeted bike distribution and collection strategies. The `day_of_week` feature is highly useful in segmenting demand and anticipating rider behavior.

In [ ]:
# Add a flag for Weekend vs Weekday
clean_df['is_weekend'] = clean_df['started_at'].dt.dayofweek // 5 == 1
# Create a new column with descriptive string labels for the legend
clean_df['day_type'] = clean_df['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

# Plotting the comparison
plt.figure(figsize=(14, 6))
sns.lineplot(data=clean_df, x='hour', y='duration_mins', hue='day_type', estimator=len)
plt.title('Demand Patterns: Weekdays vs. Weekends')
plt.ylabel('Number of Trips')
plt.xlabel('Hour of Day')
# Let seaborn handle the legend automatically for consistent colors and labels
# plt.legend(title='Is Weekend?', labels=['Weekday', 'Weekend']) # Removed manual legend
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Identify the Top 10 most active stations
top_stations = clean_df['start_station_name'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_stations.plot(kind='barh', color='skyblue')
plt.title('Top 10 Most Popular Start Stations')
plt.xlabel('Total Departures')
plt.gca().invert_yaxis() # Highest on top
plt.show()

In [ ]:
# Identify the Top 10 most active end stations
top_end_stations = clean_df['end_station_name'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_end_stations.plot(kind='barh', color='lightcoral')
plt.title('Top 10 Most Popular End Stations')
plt.xlabel('Total Arrivals')
plt.gca().invert_yaxis() # Highest on top
plt.show()

In [ ]:
# --- STEP 4: MASTER NET FLOW AGGREGATION & COORDINATE MAPPING ---

# 1. Aggregate Outflows (Departures)
outflows = clean_df.groupby(['start_station_name', 'hour']).size().reset_index(name='outflow')
outflows.columns = ['station', 'hour', 'outflow']

# 2. Aggregate Inflows (Arrivals)
inflows = clean_df.groupby(['end_station_name', 'hour']).size().reset_index(name='inflow')
inflows.columns = ['station', 'hour', 'inflow']

# 3. Create the net_flow_df by merging Inflows and Outflows
net_flow_df = pd.merge(outflows, inflows, on=['station', 'hour'], how='outer').fillna(0)
net_flow_df['net_demand'] = net_flow_df['inflow'] - net_flow_df['outflow']

# 4. Create a ROBUST Coordinate Lookup (from both start and end stations)
start_coords = clean_df[['start_station_name', 'start_lat', 'start_lng']].rename(
    columns={'start_station_name': 'station', 'start_lat': 'lat', 'start_lng': 'lng'}
)
end_coords = clean_df[['end_station_name', 'end_lat', 'end_lng']].rename(
    columns={'end_station_name': 'station', 'end_lat': 'lat', 'end_lng': 'lng'}
)

# Combine and average to get one lat/lng per station name
coords_lookup = pd.concat([start_coords, end_coords]).dropna(subset=['station']).groupby('station')[['lat', 'lng']].mean().reset_index()

# 5. Merge the coordinates into our final dataframe
net_flow_df = pd.merge(net_flow_df, coords_lookup, on='station', how='left')

# Check the result
print("Net Flow Table successfully created!")
print(f"Total stations processed: {net_flow_df['station'].nunique()}")
display(net_flow_df.head())

In [ ]:
def plot_city_imbalance(target_hour):
    # Filter for the specific hour (e.g., 8 AM Rush)
    hour_data = net_flow_df[net_flow_df['hour'] == target_hour]

    plt.figure(figsize=(12, 10))

    # We use a scatter plot where the color is the demand
    # vmin and vmax ensure 0 is the 'white' center of the color scale
    limit = max(abs(hour_data['net_demand'].min()), abs(hour_data['net_demand'].max()))

    scatter = plt.scatter(
        hour_data['lng'],
        hour_data['lat'],
        c=hour_data['net_demand'],
        cmap='RdBu',
        s=abs(hour_data['net_demand']) * 10, # Size proportional to imbalance
        alpha=0.7,
        vmin=-limit,
        vmax=limit
    )

    plt.colorbar(scatter, label='Net Demand (Arrivals - Departures)')
    plt.title(f'City-wide Station Imbalance at {target_hour}:00')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.show()

# Visualize the 8 AM Morning Commute
plot_city_imbalance(8)

# Visualize the 5 PM Evening Commute
plot_city_imbalance(17)

Feature Engineering


In [ ]:
import pandas as pd
import numpy as np

# --- 1. PREPARING THE COMPLETE TIME-SERIES GRID (RE-INDEXING) ---
# We need a row for every station, every day, and every hour to make lags accurate.

# Get unique stations and the date range from your clean_df
stations = net_flow_df['station'].unique()
dates = pd.to_datetime(clean_df['started_at']).dt.date.unique()
hours = range(24)

# Create a MultiIndex of all combinations (Product of Stations x Dates x Hours)
multi_index = pd.MultiIndex.from_product(
    [stations, dates, hours],
    names=['station', 'date', 'hour']
)

# Create the master grid
feature_df = pd.DataFrame(index=multi_index).reset_index()
feature_df['date'] = pd.to_datetime(feature_df['date'])

# Merge with your existing net_flow_df (from your previous EDA work)
# Note: Ensure net_flow_df has a 'date' column extracted from started_at
if 'date' not in net_flow_df.columns:
    # Assuming net_flow_df was built from clean_df
    # We add date back to net_flow_df if it's missing
    # This assumes we are working with the hourly aggregated data
    pass

# Merging current net_flow data into the grid
# Fill NaNs with 0 (No activity = 0 demand)
final_df = pd.merge(
    feature_df,
    net_flow_df[['station', 'hour', 'net_demand', 'lat', 'lng']],
    on=['station', 'hour'],
    how='left'
).fillna({'net_demand': 0})

# Fill Lat/Lng for the newly created empty rows using the lookup we built in EDA
final_df = final_df.drop(columns=['lat', 'lng']).merge(coords_lookup, on='station', how='left')

# 2. CYCLICAL TIME ENCODING
# Convert hour (0-23) into Sine and Cosine to represent the circular nature of time
final_df['hour_sin'] = np.sin(2 * np.pi * final_df['hour'] / 24)
final_df['hour_cos'] = np.cos(2 * np.pi * final_df['hour'] / 24)

# Day of week encoding (0=Monday, 6=Sunday)
final_df['day_of_week'] = final_df['date'].dt.dayofweek
final_df['day_sin'] = np.sin(2 * np.pi * final_df['day_of_week'] / 7)
final_df['day_cos'] = np.cos(2 * np.pi * final_df['day_of_week'] / 7)

# --- 3. MULTI-SCALE LAG FEATURES ---
# Sort by station and time to ensure shift() works correctly
final_df = final_df.sort_values(['station', 'date', 'hour'])

# Create Lags (T-1, T-2, T-3, T-24)
lags = [1, 2, 3, 24]
for lag in lags:
    final_df[f'lag_{lag}h'] = final_df.groupby('station')['net_demand'].shift(lag)

# 4. ROLLING WINDOW STATISTICS
# 3-hour moving average of demand
final_df['rolling_mean_3h'] = final_df.groupby('station')['net_demand'].transform(
    lambda x: x.shift(1).rolling(window=3).mean()
)

# --- 5. CLEANUP ---
# Drop the first 24 hours of data for each station because they won't have complete lag features
final_df = final_df.dropna()

print("Feature Engineering Complete!")
print(f"Final dataset shape: {final_df.shape}")
print("\nNew Features Created:")
print(final_df[['station', 'hour', 'net_demand', 'hour_sin', 'hour_cos', 'lag_1h', 'lag_24h']].head())

In [ ]:
import folium
from folium.features import DivIcon

# 1. Identify Top 3 Start and End Stations
top_3_starts_names = clean_df['start_station_name'].value_counts().nlargest(3).index
top_3_ends_names = clean_df['end_station_name'].value_counts().nlargest(3).index

# 2. Initialize the map
map_center = [clean_df['start_lat'].mean(), clean_df['start_lng'].mean()]
m = folium.Map(location=map_center, zoom_start=12)

# 3. Feature Groups for the Legend
start_layer = folium.FeatureGroup(name='Start Stations (Blue)')
end_layer = folium.FeatureGroup(name='End Stations (Red)')
top_labels_layer = folium.FeatureGroup(name='Top 3 Station Names')

# 4. Add All Stations (Circles)
unique_starts = clean_df.drop_duplicates(subset=['start_station_name']).dropna(subset=['start_lat', 'start_lng'])
for _, row in unique_starts.iterrows():
    folium.CircleMarker(
        location=[row['start_lat'], row['start_lng']],
        radius=3, color='#3186cc', fill=True, fill_opacity=0.3
    ).add_to(start_layer)

unique_ends = clean_df.drop_duplicates(subset=['end_station_name']).dropna(subset=['end_lat', 'end_lng'])
for _, row in unique_ends.iterrows():
    folium.CircleMarker(
        location=[row['end_lat'], row['end_lng']],
        radius=3, color='#cc3131', fill=True, fill_opacity=0.3
    ).add_to(end_layer)

# 5. Add Permanent Labels for Top 3 Starts
for name in top_3_starts_names:
    row = clean_df[clean_df['start_station_name'] == name].iloc[0]
    # The Marker for the icon
    folium.Marker(
        location=[row['start_lat'], row['start_lng']],
        icon=folium.Icon(color='blue', icon='star')
    ).add_to(top_labels_layer)

    # The DivIcon for the permanent Text Label
    folium.Marker(
        location=[row['start_lat'], row['start_lng']],
        icon=DivIcon(
            icon_size=(250,36),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12pt; color: blue; font-weight: bold; background-color: white; border: 1px solid black; padding: 2px; width: fit-content;">{name}</div>',
        )
    ).add_to(top_labels_layer)

# 6. Add Permanent Labels for Top 3 Ends
for name in top_3_ends_names:
    row = clean_df[clean_df['end_station_name'] == name].iloc[0]
    folium.Marker(
        location=[row['end_lat'], row['end_lng']],
        icon=folium.Icon(color='red', icon='star')
    ).add_to(top_labels_layer)

    folium.Marker(
        location=[row['end_lat'], row['end_lng']],
        icon=DivIcon(
            icon_size=(250,36),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12pt; color: red; font-weight: bold; background-color: white; border: 1px solid black; padding: 2px; width: fit-content;">{name}</div>',
        )
    ).add_to(top_labels_layer)

# 7. Finalize Map
start_layer.add_to(m)
end_layer.add_to(m)
top_labels_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Columns that exist in final_df (post-aggregation & feature engineering)
numerical_cols = [
    'net_demand',
    'hour',
    'hour_sin', 'hour_cos',
    'day_of_week',
    'day_sin', 'day_cos',
    'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h',
    'rolling_mean_3h',
    'lat', 'lng'
]

# Filter to only columns that actually exist (safety check)
available_cols = [col for col in numerical_cols if col in final_df.columns]
df_for_corr = final_df[available_cols].copy()

# Compute the correlation matrix
corr_matrix = df_for_corr.corr()

# Plot the heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={"size": 9},
    square=True
)
plt.title("Correlation Heatmap of Engineered Features vs. Net Demand", fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import folium

# 1. Identify Top 3 using your cleaned dataframe
top_3_starts_idx = clean_df['start_station_name'].value_counts().nlargest(3).index
top_3_ends_idx = clean_df['end_station_name'].value_counts().nlargest(3).index

# Filter and drop duplicates/NaNs
df_top_starts = clean_df[clean_df['start_station_name'].isin(top_3_starts_idx)].drop_duplicates('start_station_name').dropna(subset=['start_lat', 'start_lng'])
df_top_ends = clean_df[clean_df['end_station_name'].isin(top_3_ends_idx)].drop_duplicates('end_station_name').dropna(subset=['end_lat', 'end_lng'])

# 2. Initialize Map
if not df_top_starts.empty:
    map_center = [df_top_starts['start_lat'].iloc[0], df_top_starts['start_lng'].iloc[0]]
else:
    map_center = [51.5074, -0.1278] # Generic fallback

m = folium.Map(location=map_center, zoom_start=13)

# 3. Create Feature Groups (These act as the "Checkmark" categories in the legend)
start_group = folium.FeatureGroup(name='Top 3 Start Stations (Blue)')
end_group = folium.FeatureGroup(name='Top 3 End Stations (Red)')

# 4. Add Start Stations to their group
for _, row in df_top_starts.iterrows():
    folium.Marker(
        location=[row['start_lat'], row['start_lng']],
        popup=folium.Popup(f"<b>Station:</b> {row['start_station_name']}", max_width=200),
        tooltip=row['start_station_name'],
        icon=folium.Icon(color='blue', icon='ok', prefix='glyphicon') # 'ok' is a checkmark icon
    ).add_to(start_group)

# 5. Add End Stations to their group
for _, row in df_top_ends.iterrows():
    folium.Marker(
        location=[row['end_lat'], row['end_lng']],
        popup=folium.Popup(f"<b>Station:</b> {row['end_station_name']}", max_width=200),
        tooltip=row['end_station_name'],
        icon=folium.Icon(color='red', icon='ok', prefix='glyphicon')
    ).add_to(end_group)

# 6. Add groups to map
start_group.add_to(m)
end_group.add_to(m)

# 7. Add Layer Control (This provides the Legend/Checkmark interface)
folium.LayerControl(collapsed=False).add_to(m)

# Display the map
m

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Redefine top_3_starts as a Series object
top_3_starts = clean_df['start_station_name'].value_counts().nlargest(3)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_3_starts.index, y=top_3_starts.values, palette='viridis')
plt.title('Top 3 Most Popular Start Stations')
plt.xlabel('Start Station Name')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Redefine top_3_ends as a Series object
top_3_ends = clean_df['end_station_name'].value_counts().nlargest(3)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_3_ends.index, y=top_3_ends.values, palette='magma')
plt.title('Top 3 Most Popular End Stations')
plt.xlabel('End Station Name')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Calculate the counts
user_counts = clean_df['member_casual'].value_counts()

# 2. Define a function to show both percentage and count
def label_function(pct, allvals):
    absolute = int(round(pct/100.*sum(allvals)))
    return f"{pct:.1f}%\n({absolute:,})"

# 3. Define colors (Blue for Member, Red for Casual)
colors = ['#3186cc', '#cc3131']

# 4. Create the pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    user_counts,
    labels=user_counts.index,
    autopct=lambda pct: label_function(pct, user_counts), # Use the custom function
    startangle=140,
    colors=colors,
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

# 5. Add title and formatting
plt.title('Distribution of Member vs Casual Riders', fontsize=15, fontweight='bold')
plt.axis('equal')

# Display the plot
plt.show()